# ProtoPNet

In [ ]:
import os
import hydra
import torch
import shutil
import warnings
from torch.optim import Adam
from omegaconf import OmegaConf
from utils import check_dir
from tes_gnnNets import *
from dataset import get_dataset, get_dataloader
import torch.nn.functional as F
from torch.optim.lr_scheduler import MultiStepLR
import logging
import networkx as nx
import torch_geometric
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(config_path="tes_config", job_name="test_app")
cfg = compose(config_name="config", overrides=["datasets=NCI1", "seed=0","models.param.ba_2motifs.num_basis_per_class=10"]) #here put the same settings used for the train
#print(OmegaConf.to_yaml(cfg))


In [ ]:
config = cfg

In [ ]:
config.models.gnn_saving_dir = 'tes_checkpoints'
config.models.param = config.models.param[config.datasets.dataset_name]

if torch.cuda.is_available():
    device = torch.device('cuda', index=config.device_id)
else:
    device = torch.device('cpu')

dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                      dataset_name=config.datasets.dataset_name)
dataset.data.x = dataset.data.x.float()
dataset.data.y = dataset.data.y.squeeze().long()
if config.models.param.graph_classification:
    dataloader_params = {'batch_size': 1,
                         'random_split_flag': config.datasets.random_split_flag,
                         'data_split_ratio': config.datasets.data_split_ratio,
                         'seed': config.seed}

In [ ]:
!ls tes_checkpoints

In [ ]:
dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                          dataset_name=config.datasets.dataset_name)

In [ ]:
dataloader = get_dataloader(dataset, **dataloader_params)

In [ ]:
model = get_gnnNets(dataset.num_node_features, dataset.num_classes, config.models)

In [ ]:
f"outputs_tes/datasets={config.datasets.dataset_name},device_id=0,models.param.{config.datasets.dataset_name}.num_basis_per_class=10,models=gcn,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gcn_{config.models.param.num_basis_per_class}_3l_best.pth"

In [ ]:
saved = torch.load(f"outputs_tes_old/datasets={config.datasets.dataset_name},device_id=0,models.param.{config.datasets.dataset_name}.num_basis_per_class=10,models=gcn,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gcn_{config.models.param.num_basis_per_class}_3l_best.pth") #here the path to the saved model
state_dict = saved['net']
basis_data = saved['basis_data']
model.load_state_dict(state_dict)
model.basis_data = basis_data

In [ ]:
model.basis_concepts #these are the vectors of the prototypes

In [ ]:
model.classifier_weights

In [ ]:
model.eval()
for b in dataloader['test']:
    print(b.y, model(b.x.float(), b.edge_index)[0].argmax(-1))

In [ ]:
train_loader = dataloader['train']

In [ ]:
model.classifier_weights

In [ ]:
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
fig, ax = plt.subplots(figsize=(6, 6))
basis_id=0 #select prototype you want to visualize
data = model.basis_data[basis_id].to('cpu')
b = Batch.from_data_list([data])
out = model(data=b)
logit, embs, sims = out
node_id = sims[:,basis_id].argmax().item()
nodes = torch_geometric.utils.k_hop_subgraph(node_id, 3, data.edge_index)[0].tolist()
import copy
for p in model.parameters():
    p.requires_grad = True
data2 = copy.deepcopy(data)
b = Batch.from_data_list([data2])
out = model(data=b)
logit, embs, sims = out
x = sims[:, basis_id].detach().numpy()
x = (x-x.min())/(x.max()-x.min())
cmap = [x[i] if i in nodes else 0 for i in range(data.num_nodes)]
g = torch_geometric.utils.to_networkx(data, to_undirected=True)
pos = nx.kamada_kawai_layout(g)
nx.draw(g, node_color=cmap, pos=pos, cmap='Blues', edgecolors='black', ax=ax)
nx.draw(g.subgraph([node_id]), pos=pos, node_color='#12306b', cmap='Blues', edgecolors='red', ax=ax)

# Set common labels
fig.tight_layout(pad=5)
plt.show()

In [ ]:
def set_config_ex(dataset="ba_2motifs", seed=3, num_basis_per_class=10, config_name="tes", model_name="gcn"):
    hydra.core.global_hydra.GlobalHydra.instance().clear()
    initialize(config_path=config_name + "_" + "config" , job_name="test_app")
    cfg = compose(config_name="config", overrides=[f"datasets={dataset}", f"seed={seed}", f"models={model_name}"])
#     print(OmegaConf.to_yaml(cfg))
    config = cfg
    config.models.gnn_saving_dir = config_name + "_" + 'checkpoints'
    config.models.param = config.models.param[config.datasets.dataset_name]

    if torch.cuda.is_available():
        device = torch.device('cuda', index=config.device_id)
    else:
        device = torch.device('cpu')

    dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                          dataset_name=config.datasets.dataset_name)
    dataset.data.x = dataset.data.x.float()
    dataset.data.y = dataset.data.y.squeeze().long()
    if config.models.param.graph_classification:
        dataloader_params = {'batch_size': 1,
                             'random_split_flag': config.datasets.random_split_flag,
                             'data_split_ratio': config.datasets.data_split_ratio,
                             'seed': config.seed}
    dataset = get_dataset(dataset_root=config.datasets.dataset_root,
                          dataset_name=config.datasets.dataset_name)
    dataloader = get_dataloader(dataset, **dataloader_params)
    model = get_gnnNets(dataset.num_node_features, dataset.num_classes, config.models)
    saved = torch.load(f"outputs_tes/datasets={config.datasets.dataset_name},device_id=2,models=gcn,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gcn_{config.models.param.num_basis_per_class}_3l_latest.pth", map_location='cuda:0') #here the path to the saved model
    # saved = torch.load(f"outputs_tes_old/datasets={config.datasets.dataset_name},device_id=0,models.param.{config.datasets.dataset_name}.num_basis_per_class=10,models=gcn,seed={config.seed}/tes_checkpoints/{config.seed}/{config.datasets.dataset_name}_from_None/gcn_{config.models.param.num_basis_per_class}_3l_latest.pth", map_location='cuda:0') #here the path to the saved model
    state_dict = saved['net']
    basis_data = saved['basis_data']
    model.load_state_dict(state_dict)
    model.basis_data = basis_data
    return config, dataset, dataloader, model

In [ ]:
config, dataset, dataloader, model = set_config_ex(dataset=dataset_name, seed=i, model_name="gcn", config_name="tes")

In [ ]:
l = []
seeds = [8971, 85688, 9467, 32830, 28689, 94845, 69840, 50883, 74177, 79585, 1055, 75631, 6825, 93188, 95426, 54514, 31467, 70597, 71149, 81994]
dataset_name = 'MUTAG'
for i in seeds:
    try:
        config, dataset, dataloader, model = set_config_ex(dataset=dataset_name, seed=i, model_name="gcn", config_name="tes")
    except:
        continue
    preds, accs = [], []
    for batch in dataloader['test']:
        batch = batch
        batch_preds, _, _ = model(batch.x.float(), batch.edge_index)
        batch_preds = batch_preds.argmax(-1)
        preds.append(batch_preds)
        accs.append(batch_preds == batch.y)
    preds = torch.cat(preds, dim=-1)
    test_acc = torch.cat(accs, dim=-1).float().mean().item()
    l.append(test_acc)

In [ ]:
np.mean(l)

In [ ]:
np.std(l)

In [ ]:
def explain(model, data, sparsity=0.8, c=None, protop=True):
    out, embs, cosines=model(data.x.float(), data.edge_index)    
    y_hat=c
    if y_hat is None:
        y_hat=out.argmax(-1).item()
    mask=np.zeros(len(data.x))
    if True:
        max_nodes = cosines[:,model.num_basis_per_class*y_hat:model.num_basis_per_class*(y_hat+1)].argmax(axis=0).tolist()
        for i in range(model.num_basis_per_class*y_hat,model.num_basis_per_class*(y_hat+1)):
            try:
                to_mask = torch_geometric.utils.k_hop_subgraph(max_nodes[i%model.num_basis_per_class], 3, data.edge_index)[0].tolist()
                cosines_i = cosines[max_nodes[i%model.num_basis_per_class],i].item()
                mask[to_mask] += model.classifier_weights[i, y_hat].detach().numpy()*cosines_i
            except:pass
    else:
        for i in range(len(data.x)):
            mask[i]+= cosines[i,model.num_basis_per_class*y_hat:model.num_basis_per_class*(y_hat+1)].sum().item()
#     if protop: mask=torch.log(mask+1)/(mask+1e-8)
#     mask = mask*model.classifier_weights[i]
    mask=(mask-mask.min())/(mask.max()-mask.min()+1e-8)
    p=np.percentile(mask, 100*sparsity)
    mask[mask<p]=0
    edge_mask=np.zeros(data.edge_index.shape[1])
    for i, (a,b) in enumerate(data.edge_index.T):
        edge_mask[i]=(mask[a]+mask[b])/2
    edge_mask[edge_mask>0]=1
    return mask#, edge_mask
    

In [ ]:
dataset

In [ ]:
dataset[0].y

In [ ]:
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
fig, ax = plt.subplots(figsize=(6, 6))
data = dataset[500]
b = Batch.from_data_list([data])
out = model(data=b)
logit, embs, sims = out

print((logit.softmax(-1)[0,0]).item()*100)
e = explain(model, data,c=0) - explain(model, data,c=1)
# print(e)
# cmap = [x[i] if i in nodes else 0 for i in range(data.num_nodes)]
g = torch_geometric.utils.to_networkx(data, to_undirected=True)
pos = nx.kamada_kawai_layout(g)

labels = {node: str(node) for node in g.nodes()}
nx.draw(g, node_color=e, pos=pos, cmap='coolwarm', vmin=-1, vmax=1,edgecolors='black', ax=ax, labels=labels)

# # Set common labels
fig.tight_layout(pad=5)
plt.show()

In [ ]:
batch

In [ ]:
e.round()

In [ ]:
from torch_geometric.data import Data
from torch_geometric.utils import subgraph
data_orig = data
cond = torch.tensor(e.round() <= 0)
print(cond)
masked_edge_index, _ = subgraph(cond, data_orig.edge_index, relabel_nodes=True, num_nodes=data_orig.x.shape[0])
data_mask = Data(x=data.x[cond], y=data.y, edge_index=masked_edge_index, edge_weight=None, batch=torch.zeros_like(cond)) 
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
fig, ax = plt.subplots(figsize=(6, 6))
b = Batch.from_data_list([data_mask])
out = model(data=b)
logit, embs, sims = out
print((logit.softmax(-1)[0,0]).item()*100)
e = explain(model, data_mask,c=0)-explain(model, data_mask,c=1)
print(e)
# cmap = [x[i] if i in nodes else 0 for i in range(data.num_nodes)]
g = torch_geometric.utils.to_networkx(data_mask, to_undirected=True)

pos = nx.kamada_kawai_layout(g)
labels = {node: str(node) for node in g.nodes()}
nx.draw(g, node_color=e, pos=pos, cmap='coolwarm', vmin=-1, vmax=1,edgecolors='black', ax=ax, labels=labels)

# # Set common labels
fig.tight_layout(pad=5)
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
def duplicate_and_attach(graph, mask):
    import copy
    graph = copy.deepcopy(graph)
    # Step 1: Identify the indices of the true values in the mask
    true_indices = torch.nonzero(mask).view(-1)

    if len(true_indices) == 0:
        # No true values in the mask, nothing to duplicate and attach
        return graph

    # Step 2: Randomly pick an index from the true values of the mask
    selected_index = torch.randint(len(true_indices), (1,)).item()
    selected_node_index = true_indices[selected_index]

    # Step 3: Duplicate the node corresponding to the selected index
    duplicated_node = graph.x[selected_node_index].clone()
    graph.x = torch.cat([graph.x, duplicated_node.view(1, -1)], dim=0)

    # Step 4: Attach the duplicated node to another random node in the graph
    attached_node_index = torch.randint(len(true_indices), (1,)).item()
    graph.edge_index = torch.cat([graph.edge_index, torch.tensor([[selected_node_index, len(graph.x) - 1]], dtype=torch.long).t()], dim=1)

    # Update the 'batch' parameter
    graph.batch = torch.cat([graph.batch, torch.zeros(1, dtype=torch.long)])

    return graph
import copy
def expl_res(model, dataloader):
    expl_true = []
    expl_pred = []
    expl_stab_true = []
    expl_stab_pred = []
    from torch_geometric.data import Data
    from torch_geometric.utils import subgraph
    test_loader = dataloader['test']
    fidelity = []
    for i, data in enumerate(test_loader):
        # if data.y.item() == 1: 
            # continue
        try:
            data.batch=torch.zeros(data.x.shape[0])
            out = model(data.x.float(), data.edge_index)[0]
            c = out.argmax(-1).item()
            # expl = torch.tensor(explain(model,data,c=0, sparsity=0.9))>0
            expl = torch.tensor(explain(model,data,c=c, sparsity=0.9))>0
            # print(expl)
            # expl = (1-expl).int()
            batch = torch.zeros(data.x.shape[0]).to(data.x.device).long()
            data_orig = Data(x=data.x, y=data.y,   edge_index=data.edge_index, edge_weight=None, batch=batch)
            masked_edge_index, _ = subgraph(expl==0, data_orig.edge_index, relabel_nodes=True, num_nodes=data_orig.x.shape[0])
            data_mask = Data(x=data.x[expl==0], y=data.y, edge_index=masked_edge_index, edge_weight=None, batch=batch[expl==0]) 
            fidelity.append(model(data_orig.x.float(), data_orig.edge_index)[0].softmax(-1)[0,0].item() - model(data_mask.x.float(), data_mask.edge_index)[0].softmax(-1)[0,0].item())
            # print(self.model(data_orig)[0].softmax(-1)[0,0].item(),  self.model(data_mask)[0].softmax(-1)[0,0].item())

            data_stab = duplicate_and_attach(copy.deepcopy(data_orig), expl==0)
            
            # stab_mask = (expl == 1) | ((expl == 0) & (torch.rand(data.x.shape[0])>0.2).to(expl.device))
            # print((expl == 1))
            # print((expl == 0 & (torch.rand(data.x.shape[0])>0.8).to(expl.device)))
            # print(stab_mask)
            # print(stab_mask)
            # print(stab_mask)
            # print(data[i].node_label)
            # stab_edge_index, _ = subgraph(stab_mask, data_orig.edge_index, relabel_nodes=True, num_nodes=data_orig.x.shape[0])
            # data_stab = Data(x=data.x[stab_mask], y=data.y, edge_index=stab_edge_index, edge_weight=None, batch=batch[stab_mask])
            expl_stab = torch.tensor(explain(model,data_stab,c=c, sparsity=0.9))>0
            # print(expl_stab.detach().cpu().numpy().tolist())
            # print(expl[stab_mask].detach().cpu().numpy().tolist())

            # print(expl_stab.detach().cpu().numpy().tolist())
            # print(expl[stab_mask].detach().cpu().numpy().tolist())
            expl_stab_true.extend(expl_stab[:-1].detach().cpu().numpy().tolist())
            expl_stab_pred.extend(expl.detach().cpu().numpy().tolist())

#             for a, b in zip(expl, data.node_label):
#                 expl_true.append(b.item())
#                 expl_pred.append(a.item())
        except Exception as e: print(e)
    return np.mean(fidelity),accuracy_score(expl_stab_true, expl_stab_pred),accuracy_score(expl_true, expl_pred)

In [ ]:
expl_res(model, dataloader)

In [ ]:
def expl_res1(model, dataloader):
    expl_true = []
    expl_pred = []
    expl_stab_true = []
    expl_stab_pred = []
    from torch_geometric.data import Data
    from torch_geometric.utils import subgraph
    test_loader = dataloader['test']
    fidelity = []
    for i, data in enumerate(test_loader):
        # if data.y.item() == 1: 
            # continue
        # try:
        data.batch=torch.zeros(data.x.shape[0])
        out = model(data.x.float(), data.edge_index)[0]
        c = out.argmax(-1).item()
        expl = torch.tensor(explain(model,data,c=c, sparsity=0.9))>0
        # print(expl)
        # expl = (1-expl).int()
        batch = torch.zeros(data.x.shape[0]).to(data.x.device).long()
        data_orig = Data(x=data.x, y=data.y,   edge_index=data.edge_index, edge_weight=None, batch=batch)
        masked_edge_index, _ = subgraph(expl==0, data_orig.edge_index, relabel_nodes=True, num_nodes=data_orig.x.shape[0])
        data_mask = Data(x=data.x[expl==0], y=data.y, edge_index=masked_edge_index, edge_weight=None, batch=batch[expl==0]) 
        fidelity.append(model(data_orig.x.float(), data_orig.edge_index)[0].softmax(-1)[0,0].item() - model(data_mask.x.float(), data_mask.edge_index)[0].softmax(-1)[0,0].item())
        # print(self.model(data_orig)[0].softmax(-1)[0,0].item(),  self.model(data_mask)[0].softmax(-1)[0,0].item())

        
        
        
        stab_mask = (expl == 1) | (expl == 0 & (torch.rand(data.y.shape)>0.2).to(expl.device))
        # print(stab_mask)
        # print(data[i].node_label)
        stab_edge_index, _ = subgraph(stab_mask, data_orig.edge_index, relabel_nodes=True, num_nodes=data_orig.x.shape[0])
        data_stab = Data(x=data.x[stab_mask], y=data.y, edge_index=stab_edge_index, edge_weight=None, batch=batch[stab_mask])
        expl_stab = torch.tensor(explain(model,data,c=c, sparsity=0.9))>0
        # print(expl_stab.detach().cpu().numpy().tolist())
        # print(expl[stab_mask].detach().cpu().numpy().tolist())
        expl_stab_true.extend(expl_stab.detach().cpu().numpy().tolist())
        expl_stab_pred.extend(expl[stab_mask].detach().cpu().numpy().tolist())

        for a, b in zip(expl, data.node_label):
            expl_true.append(b.item())
            expl_pred.append(a.item())
        # except Exception as e: print(e)
    return np.mean(fidelity),accuracy_score(expl_stab_true, expl_stab_pred),accuracy_score(expl_true, expl_pred)

In [ ]:
dataset[0]

In [ ]:
fs = []
acs = []
ss = []
l = []
dataset_name='NCI1'
for i in seeds:
    try:
        config, dataset, dataloader, model = set_config_ex(dataset=dataset_name, seed=i, model_name="gcn", config_name="tes")
    except Exception as e:
        print(e)
        continue
        
    f, s, a = expl_res(model, dataloader)
    preds, accs = [], []
    for batch in dataloader['test']:
        batch = batch
        batch_preds, _, _ = model(batch.x.float(), batch.edge_index)
        batch_preds = batch_preds.argmax(-1)
        preds.append(batch_preds)
        accs.append(batch_preds == batch.y)
    preds = torch.cat(preds, dim=-1)
    test_acc = torch.cat(accs, dim=-1).float().mean().item()
    l.append(test_acc)
    ss.append(s)
    fs.append(f)
    acs.append(a)

In [ ]:
fs = [f for il,f in zip(l, fs) if il>0.8]
acs = [a for il,a in zip(l, acs) if il>0.8]
ss = [s for il,s in zip(l, ss) if il>0.8]
l = [il for il in l if il>0.8]


In [ ]:
fs = [f for il,f in zip(l, fs) ]
acs = [a for il,a in zip(l, acs)]
ss = [s for il,s in zip(l, ss) ]
l = [il for il in l ]

In [ ]:
ss

In [ ]:
sum(ss)/len(ss)

In [ ]:
np.std(ss)

In [ ]:
np.mean(fs[:3]+fs[4:])

In [ ]:
np.mean(ss), np.std(ss)

In [ ]:
np.mean(acs), np.std(acs)

In [ ]:
np.mean(l), np.std(l)

In [ ]:
dataset[0]

In [ ]:
l